# Chương 7: Hệ đa agent: State, Message, hợp hàm và kiểm tra lỗi

    Hệ điều phối xác định với bốn vai trò (truy hồi, đồ thị, viết, kiểm tra); chưa phải nhóm LLM tự lập kế hoạch.
    Chạy cục bộ trên cùng kho tài liệu và mô hình Qwen2.5-1.5B-Instruct của Chương 5. Demo localhost của chương này ở `/agents`
    (cổng 8044). Mỗi ô đi kèm điều cần giải thích.

In [1]:
# Cell AGENT-01
import os, sys, json, time, inspect, copy
from pathlib import Path
for key in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[key] = '2'
os.environ['KERAS_BACKEND'] = 'tensorflow'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'src/tieuluan').exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
OUT = ROOT/'results/tieuluan/complete/notebooks'
OUT.mkdir(parents=True, exist_ok=True)
print('Project:', ROOT)

Project: E:\WINDOW\Project\intelligent-system-assignment-01


## 1. Mã thực của State, Message và các agent

    State chứa câu hỏi, truy vấn tiếng Anh, evidence, graph_context, answer, status và messages. Message ghi sender, receiver và payload.
    Bốn phép chuyển trạng thái hợp thành S4 = Verifier ∘ Writer ∘ Graph ∘ Retrieval (S0).
    Ô tiếp theo định nghĩa đầy đủ các lớp, không chỉ import tên. Vai trò đời thường: Agent truy hồi là người đi tìm tài liệu,
    Agent đồ thị là người tra nguồn của từng đoạn, Agent viết là người soạn câu trả lời, Agent kiểm tra là người soát nguồn trích dẫn.

In [2]:
# Cell AGENT-02
"""Bounded, role-based multi-agent demonstration with explicit messages."""
from dataclasses import dataclass, field


@dataclass
class Message:
    sender: str
    receiver: str
    payload: dict


@dataclass
class State:
    question: str
    domain: str
    report: str | None = None
    evidence: list = field(default_factory=list)
    graph_context: list = field(default_factory=list)
    answer: str = ''
    status: str = 'pending'
    messages: list = field(default_factory=list)
    query: str = ''
    language: str = ''
    english_answer: str = ''

    def send(self, sender, receiver, **payload):
        self.messages.append(Message(sender, receiver, payload))


class RetrievalAgent:
    def __init__(self, engine):
        self.engine = engine

    def act(self, state):
        # Câu tiếng Việt được dịch sang truy vấn tiếng Anh vì kho tài liệu bằng tiếng Anh.
        if hasattr(self.engine, 'to_english_query'):
            state.query, state.language = self.engine.to_english_query(state.question)
        else:
            state.query, state.language = state.question, 'en'
        state.evidence = self.engine.retrieve(
            state.query, state.domain, k=3, report=state.report)
        state.send('retrieval', 'graph',
                   ids=[e['id'] for e in state.evidence])


class GraphAgent:
    def __init__(self, graphs):
        self.graphs = graphs

    def act(self, state):
        graph = self.graphs.get(state.domain)
        if graph is not None:
            for item in state.evidence:
                node = 'passage:' + item['id']
                if node in graph:
                    state.graph_context.append({
                        'id': item['id'],
                        'parents': list(graph.predecessors(node))})
        state.send('graph', 'writer',
                   matched=len(state.graph_context))


class WriterAgent:
    def __init__(self, generator):
        self.generator = generator

    def act(self, state):
        from src.chapter04.rag import INSUFFICIENT_EN, INSUFFICIENT_VI, lexical_support
        refuse = INSUFFICIENT_VI if state.language == 'vi' else INSUFFICIENT_EN
        if not state.evidence or not lexical_support(state.query or state.question, state.evidence):
            state.status = 'insufficient_evidence'
            state.answer = refuse
        elif hasattr(self.generator, 'generate_full'):
            full = self.generator.generate_full(
                state.question, state.evidence, state.language or None,
                state.query or None)
            state.answer = full['answer']
            state.english_answer = full['english_answer'] or ''
            if full['insufficient']:
                state.status = 'insufficient_evidence'
        else:
            state.answer = self.generator.generate(
                state.question, state.evidence)
        state.send('writer', 'verifier', answer=state.answer)


class VerifierAgent:
    def __init__(self, corpora):
        self.allowed = {d: {x['id'] for x in c}
                        for d, c in corpora.items()}

    def act(self, state):
        ids = [e['id'] for e in state.evidence]
        valid = bool(ids) and all(
            cid in self.allowed[state.domain] for cid in ids)
        if state.status == 'insufficient_evidence' and state.answer.strip():
            pass  # Agent viết đã từ chối: giữ nguyên câu từ chối.
        elif not valid or not state.answer.strip():
            state.answer = ('Không đủ bằng chứng.' if state.language == 'vi'
                            else 'Insufficient evidence')
            state.status = 'insufficient_evidence'
        else:
            state.status = 'generated_unverified'
        state.send('verifier', 'coordinator', citation_ids_valid=valid,
                   semantic_support_checked=False, status=state.status)


class MultiAgentTeam:
    def __init__(self, engine, graphs):
        self.agents = [RetrievalAgent(engine), GraphAgent(graphs),
                       WriterAgent(engine.generator),
                       VerifierAgent(engine.corpora)]

    def ask(self, question, domain, report=None):
        if domain not in ('diabetes', 'finance'):
            raise ValueError('Unsupported domain')
        if not question.strip():
            raise ValueError('Empty question')
        state = State(question, domain, report)
        for agent in self.agents:  # Four bounded turns; no retry loop.
            agent.act(state)
        return state

## 2. Khởi tạo phụ thuộc

    `rag()` nạp kho tài liệu, hai bộ truy hồi và mô hình sinh; `load_graph` nạp hai đồ thị (tài chính và tài liệu NIDDK);
    MultiAgentTeam gắn từng phụ thuộc cho vai trò cần dùng. Chỉ dùng một mô hình sinh chung, không gọi bốn mô hình độc lập.

In [3]:
# Cell AGENT-03
from deployment.chapter04.app import rag, agent, summarize
from src.chapter04.knowledge import load_graph
graphs = {'finance':load_graph(ROOT/'knowledge_graph/chapter04/finance_graph.json.gz'),
          'diabetes':load_graph(ROOT/'knowledge_graph/chapter04/diabetes_documents_graph.json.gz')}
engine = rag(); engine.generator.load()
team = MultiAgentTeam(engine,graphs)
print([type(a).__name__ for a in team.agents])

['RetrievalAgent', 'GraphAgent', 'WriterAgent', 'VerifierAgent']


## 3. Quan sát hợp hàm bằng cách chạy riêng từng agent

    Sau mỗi `act`, in câu kể lại việc agent vừa làm (hàm `summarize` của demo), số bằng chứng, ngữ cảnh đồ thị, trạng thái và thông điệp vừa tạo.
    Agent truy hồi không viết đáp án; Agent đồ thị bổ sung nguồn cha; Agent viết sinh câu trả lời; Agent kiểm tra chỉ kiểm tra mã nguồn,
    chưa xác minh ngữ nghĩa nên trạng thái cuối là `generated_unverified`.

In [4]:
# Cell AGENT-04
state = State('Xét nghiệm A1C cho biết điều gì?','diabetes')
trace = []
for worker in team.agents:
    t0 = time.perf_counter(); worker.act(state); seconds = time.perf_counter()-t0
    story, outcome = summarize(type(worker).__name__, state, worker, seconds)
    trace.append({'agent':type(worker).__name__,'kết quả':outcome,'evidence':len(state.evidence),
                  'graph':len(state.graph_context),'status':state.status,'giây':round(seconds,1)})
    message = state.messages[-1]
    print(type(worker).__name__, '=>', story)
    print('   Thông điệp:', message.sender,'->',message.receiver,message.payload)
display(pd.DataFrame(trace))
print('Câu trả lời cuối:', state.answer)
assert len(state.messages)==4
assert not state.messages[-1].payload['semantic_support_checked']

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


RetrievalAgent => Agent truy hồi đã tìm được 3 đoạn tài liệu liên quan. Vì kho tài liệu bằng tiếng Anh, nó dịch câu hỏi thành truy vấn: "What does an A1C test indicate?". Đoạn tốt nhất đến từ NIDDK · Kiểm soát bệnh tiểu đường và viết: «The A1C test shows your average blood glucose level over the last 3 months. This test is done at a health care professional’s office or lab. For most people with diabetes, their goal is an A1C level b …»
   Thông điệp: retrieval -> graph {'ids': ['managing-diabetes:21', 'tests-diagnosis:41', 'tests-diagnosis:43']}
GraphAgent => Agent đồ thị đã tra 3 đoạn trong đồ thị tri thức và biết chúng thuộc: NIDDK · Kiểm soát bệnh tiểu đường; NIDDK · Xét nghiệm và chẩn đoán. Nhờ vậy câu trả lời có thể nói rõ nguồn nào.
   Thông điệp: graph -> writer {'matched': 3}


WriterAgent => Agent viết đã soạn câu trả lời chỉ dựa trên các đoạn đó: «Xét nghiệm A1C cho thấy mức đường huyết trung bình trong 3 tháng gần nhất.»
   Thông điệp: writer -> verifier {'answer': 'Xét nghiệm A1C cho thấy mức đường huyết trung bình trong 3 tháng gần nhất.'}
VerifierAgent => Agent kiểm tra soát 3 mã nguồn trích dẫn: tất cả đều có thật trong kho tài liệu. Nhưng nó chưa so được nghĩa câu trả lời với nguồn, nên gắn nhãn "chưa kiểm chứng".
   Thông điệp: verifier -> coordinator {'citation_ids_valid': True, 'semantic_support_checked': False, 'status': 'generated_unverified'}


,agent,kết quả,evidence,graph,status,giây
0,RetrievalAgent,3 đoạn,3,0,pending,5.5
1,GraphAgent,3 đoạn khớp,3,3,pending,0.0
2,WriterAgent,đã viết xong,3,3,pending,29.1
3,VerifierAgent,"nguồn hợp lệ, chưa kiểm chứng ý nghĩa",3,3,generated_unverified,0.0


Câu trả lời cuối: Xét nghiệm A1C cho thấy mức đường huyết trung bình trong 3 tháng gần nhất.


## 4. Kiểm tra các đường lỗi

    `ask` chặn domain sai và câu hỏi rỗng. Khi report không tồn tại, truy hồi trả rỗng, Agent viết không sinh và Agent kiểm tra
    trả `insufficient_evidence`. Điểm dừng: mỗi agent chạy một lần, không có vòng lặp đến khi đồng ý.

In [5]:
# Cell AGENT-05
missing = team.ask('Find revenue','finance',report='__missing_report__')
print('Không có nguồn:',missing.status,missing.answer)
assert missing.status=='insufficient_evidence'
for question, domain in [('', 'finance'), ('test','invalid')]:
    try:
        team.ask(question,domain)
    except ValueError as exc:
        print('Đầu vào bị chặn:',exc)
    else:
        raise AssertionError('Đầu vào sai không bị chặn')

Không có nguồn: insufficient_evidence There is not enough information in the documents to answer this question.
Đầu vào bị chặn: Empty question
Đầu vào bị chặn: Unsupported domain


## 5. Câu hỏi không có trong tài liệu: từ chối "không đủ bằng chứng"

    Với câu hỏi ngoài kho tài liệu, nhóm agent phải từ chối thay vì bịa. Agent truy hồi vẫn trả về ba đoạn gần nhất, nhưng không đoạn nào chứa
    từ khóa của câu hỏi; Agent viết nhận ra điều đó và Agent kiểm tra giữ nguyên lời từ chối.

In [6]:
# Cell AGENT-06
gold = team.ask('Giá vàng hôm nay là bao nhiêu?','diabetes')
print('Trạng thái:', gold.status, '| truy vấn:', gold.query)
print('Câu trả lời:', gold.answer)
assert gold.status=='insufficient_evidence'

Trạng thái: insufficient_evidence | truy vấn: What is the current price of gold?
Câu trả lời: Không đủ thông tin trong tài liệu để trả lời câu hỏi này.


## 6. Vì sao nguồn tồn tại chưa chứng minh câu trả lời đúng?

    Chạy một câu hỏi đếm qua nhóm agent rồi đối chiếu công cụ thống kê của ToolAgent. Chỉ công cụ thống kê đọc số từ dữ liệu thật
    (CDC BRFSS 2023, 150.000 dòng). Câu mô hình viết có thể lấy một con số trong tài liệu (ví dụ số người mắc bệnh ở Hoa Kỳ) hoàn toàn
    không liên quan đến dữ liệu khảo sát. Mã nguồn trích dẫn vẫn hợp lệ, vì vậy Agent kiểm tra không bắt được lỗi này: đó là giới hạn của kiểm tra cấu trúc.

In [7]:
# Cell AGENT-07
question = 'Có bao nhiêu người trong dữ liệu khảo sát?'
generated = team.ask(question,'diabetes')
computed = agent().ask(question,'diabetes')
print('Nhóm agent viết:', generated.answer, '|', generated.status)
print('Công cụ thống kê:', computed['answer'], '|', computed['status'])
print('Nguồn id hợp lệ:', generated.messages[-1].payload['citation_ids_valid'])
payload = {'trace':trace,
    'messages':[vars(m) for m in state.messages],
    'generated_count':generated.answer,
    'computed_count':computed['answer'],
    'missing_status':missing.status,'gold_status':gold.status}
(OUT/'agents_demo.json').write_text(
    json.dumps(payload,ensure_ascii=False,indent=2),encoding='utf8')

Nhóm agent viết: 37.3 triệu | generated_unverified
Công cụ thống kê: Dữ liệu có 150.000 người trả lời khảo sát BRFSS 2023 (mỗi dòng là một người). | computed
Nguồn id hợp lệ: True


1684

## Câu hỏi trình bày

    Hãy giải thích riêng State.send, RetrievalAgent.act, GraphAgent.act, WriterAgent.act, VerifierAgent.act và MultiAgentTeam.ask.
    Chỉ ra trường State nào bị đổi ở từng bước, Message gửi cho ai và điều kiện dừng/lỗi. Tại sao không được đổi generated_unverified
    thành verified chỉ vì id hợp lệ?